In [1]:
# Install AMP Evaluation SDK
!pip install git+https://github.com/wso2/agent-manager.git@1fc4cec39c65cd5b0ca1cbd5e7a1dd54904e313a#subdirectory=libs/amp-evaluation
!pip install litellm==1.81.14
# Install Unsloth for fast 4-bit SLM inference on Kaggle GPU
!pip install unsloth -q

# Other dependencies
!pip install datasets huggingface_hub scikit-learn pandas numpy tqdm -q

!pip install opentelemetry-api==1.38.0 opentelemetry-sdk==1.38.0
!pip install google-cloud-bigquery-storage
!pip install fsspec==2026.2.0

print("All dependencies installed.")

  Cloning https://github.com/wso2/agent-manager.git (to revision 1fc4cec39c65cd5b0ca1cbd5e7a1dd54904e313a) to /tmp/pip-req-build-8gur3kzo
  Running command git clone --filter=blob:none --quiet https://github.com/wso2/agent-manager.git /tmp/pip-req-build-8gur3kzo
  Running command git rev-parse -q --verify 'sha^1fc4cec39c65cd5b0ca1cbd5e7a1dd54904e313a'
  Running command git fetch -q https://github.com/wso2/agent-manager.git 1fc4cec39c65cd5b0ca1cbd5e7a1dd54904e313a
  Running command git checkout -q 1fc4cec39c65cd5b0ca1cbd5e7a1dd54904e313a
  Resolved https://github.com/wso2/agent-manager.git to commit 1fc4cec39c65cd5b0ca1cbd5e7a1dd54904e313a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.0 

In [2]:
import amp_evaluation
import litellm
from amp_evaluation import builtin

print("amp_evaluation OK")
print("litellm OK")
print("sample evaluator:", builtin("helpfulness").info.name)


amp_evaluation OK
litellm OK
sample evaluator: helpfulness


## Cell 2 — Configuration & Dataset Access


In [3]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")  # needed to pull TRAIL

DATASET_SPLIT = "gaia"
RESULTS_DIR = "/kaggle/working"
print("HF token loaded")
print(f"Dataset split: {DATASET_SPLIT}")
print(f"Artifacts dir: {RESULTS_DIR}")


HF token loaded
Dataset split: gaia
Artifacts dir: /kaggle/working


## Cell 3 — Pull TRAIL Dataset (GAIA subset)

In [4]:
from datasets import load_dataset
import pandas as pd
import os

print("Loading TRAIL GAIA dataset from HuggingFace...")

trail_dataset = load_dataset(
    "PatronusAI/TRAIL",
    token=os.environ["HF_TOKEN"],
)

print("Available splits:", trail_dataset.keys())
print(f"Dataset features: {trail_dataset['gaia'].features}")

df_gaia = trail_dataset["gaia"].to_pandas().reset_index(drop=True)

print(f"\nTotal GAIA traces: {len(df_gaia)}")
print(f"Columns: {list(df_gaia.columns)}")

df_gaia.head(2)


Loading TRAIL GAIA dataset from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

data/gaia-00000-of-00001-33a2e72d362d688(…):   0%|          | 0.00/33.2M [00:00<?, ?B/s]

data/swe_bench-00000-of-00001-91aa04220f(…):   0%|          | 0.00/21.5M [00:00<?, ?B/s]

Generating gaia split:   0%|          | 0/117 [00:00<?, ? examples/s]

Generating swe_bench split:   0%|          | 0/31 [00:00<?, ? examples/s]

Available splits: dict_keys(['gaia', 'swe_bench'])
Dataset features: {'trace': Value('string'), 'labels': Value('string')}

Total GAIA traces: 117
Columns: ['trace', 'labels']


,trace,labels
0,"{\n ""trace_id"": ""041b7f9c8c76c2ca1a8e67c676...","{\n ""trace_id"": ""041b7f9c8c76c2ca1a8e67c676..."
1,"{\n ""trace_id"": ""4a8d094e92433f1ba1da21f602...","{\n ""trace_id"": ""4a8d094e92433f1ba1da21f602..."


## Cell 4 — Filter GAIA Subset & Inspect Structure

In [5]:
print("\n--- Sample GAIA row structure ---")
sample = df_gaia.iloc[0]

for col in df_gaia.columns:
    val = sample[col]
    preview = str(val)[:500] if val is not None else "None"
    print(f"{col}: {preview}")
    print()



--- Sample GAIA row structure ---
trace: {
    "trace_id": "041b7f9c8c76c2ca1a8e67c6769267c3",
    "spans": [
        {
            "timestamp": "2025-03-19T16:37:55.005053Z",
            "trace_id": "041b7f9c8c76c2ca1a8e67c6769267c3",
            "span_id": "ef641bfc63faffaf",
            "parent_span_id": null,
            "trace_state": "",
            "span_name": "main",
            "span_kind": "Internal",
            "service_name": "gaia-annotation-samples/app:GAIA-Samples",
            "resource_attributes": {
                

labels: {
    "trace_id": "041b7f9c8c76c2ca1a8e67c6769267c3",
    "errors": [
        {
            "category": "Tool-related",
            "location": "1832b9469b9b862d",
            "evidence": "After a search for the number of Nature research articles published in 2020 (excluding book reviews, columns, etc.), a reliable source indicates that Nature published 484 research articles that year.",
            "description": "The system hallucinated the o

## Cell 5 — Parse TRAIL Annotations into AMP Trace Objects

### Preprocess HF dataframe traces and save JSON files with ampAttributes


In [6]:
import json
import os
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

PREPROCESSED_DIR = Path("/kaggle/working/preprocessed_traces")
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def to_int(value: Any) -> Optional[int]:
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, int):
        return value
    if isinstance(value, float):
        return int(value)
    if isinstance(value, str):
        try:
            return int(float(value.strip()))
        except ValueError:
            return None
    return None


def to_float(value: Any) -> Optional[float]:
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        try:
            return float(value.strip())
        except ValueError:
            return None
    return None


def maybe_parse_json(value: Any) -> Any:
    if not isinstance(value, str):
        return value
    stripped = value.strip()
    if not stripped:
        return value
    if stripped[0] not in "[{\"":
        return value
    try:
        return json.loads(stripped)
    except json.JSONDecodeError:
        return value


def infer_kind(span_name: str, attrs: Dict[str, Any]) -> str:
    kind = str(attrs.get("openinference.span.kind", "")).strip().lower()
    if kind in {"llm", "tool", "embedding", "retriever", "agent"}:
        return kind
    if kind in {"chain", "task", "workflow", "crewaitask"}:
        return "chain"

    lower_name = span_name.lower()
    if "retriev" in lower_name or "vector" in lower_name:
        return "retriever"
    if "agent" in lower_name:
        return "agent"
    if "tool" in lower_name or "finalanswertool" in lower_name:
        return "tool"
    if "litellmmodel" in lower_name or "chat" in lower_name or "completion" in lower_name:
        return "llm"
    if "step" in lower_name or "chain" in lower_name:
        return "chain"
    return "unknown"


def extract_status_and_error(span: Dict[str, Any], attrs: Dict[str, Any]) -> Tuple[Dict[str, Any], Optional[Dict[str, Any]]]:
    status_code = str(span.get("status_code", "")).strip().lower()
    status_message = str(span.get("status_message", "")).strip()

    error_type = attrs.get("error.type")
    error_message = attrs.get("error.message") or status_message or None

    has_error = False
    if isinstance(error_type, str) and error_type.strip():
        has_error = True
    elif status_code in {"error", "failed"}:
        has_error = True

    status = {"error": has_error}
    if has_error and isinstance(error_type, str) and error_type.strip():
        status["errorType"] = error_type.strip()
    elif has_error and status_message:
        status["errorType"] = status_message
    elif has_error:
        status["errorType"] = "StatusCodeError"

    error_obj = {"message": str(error_message)} if has_error and error_message else None
    return status, error_obj


def extract_token_usage(attrs: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    input_tokens = to_int(attrs.get("llm.token_count.prompt"))
    output_tokens = to_int(attrs.get("llm.token_count.completion"))
    total_tokens = to_int(attrs.get("llm.token_count.total"))

    if input_tokens is None and output_tokens is None and total_tokens is None:
        return None

    input_tokens = input_tokens or 0
    output_tokens = output_tokens or 0
    total_tokens = total_tokens if total_tokens is not None else input_tokens + output_tokens

    return {
        "inputTokens": input_tokens,
        "outputTokens": output_tokens,
        "totalTokens": total_tokens,
    }


def parse_temperature(attrs: Dict[str, Any]) -> Optional[float]:
    raw = maybe_parse_json(attrs.get("llm.invocation_parameters"))
    if isinstance(raw, dict):
        for key in ("temperature", "temp"):
            value = to_float(raw.get(key))
            if value is not None:
                return value
    return None


def extract_llm_tools(attrs: Dict[str, Any]) -> List[Dict[str, Any]]:
    pattern = re.compile(r"^llm\.tools\.(\d+)\.tool\.json_schema$")
    bucket: Dict[int, Dict[str, Any]] = {}

    for key, value in attrs.items():
        match = pattern.match(key)
        if not match:
            continue

        index = int(match.group(1))
        parsed = maybe_parse_json(value)
        if not isinstance(parsed, dict):
            continue

        function_obj = parsed.get("function") if isinstance(parsed.get("function"), dict) else None
        source = function_obj if function_obj else parsed

        name = str(source.get("name", "")).strip()
        description = str(source.get("description", "")).strip()
        parameters_obj = source.get("parameters")

        parameters = ""
        if parameters_obj is not None:
            try:
                parameters = json.dumps(parameters_obj, ensure_ascii=False)
            except TypeError:
                parameters = str(parameters_obj)

        bucket[index] = {
            "name": name,
            "description": description,
            "parameters": parameters,
        }

    tools = []
    for index in sorted(bucket):
        tool = {k: v for k, v in bucket[index].items() if v not in ("", None)}
        if tool:
            tools.append(tool)
    return tools


def extract_message_blocks(attrs: Dict[str, Any], prefix: str) -> List[Dict[str, Any]]:
    pattern = re.compile(rf"^{re.escape(prefix)}\.(\d+)\.message\.(.+)$")
    bucket: Dict[int, Dict[str, Any]] = {}

    for key, value in attrs.items():
        match = pattern.match(key)
        if not match:
            continue
        index = int(match.group(1))
        field = match.group(2)
        bucket.setdefault(index, {})[field] = value

    messages: List[Dict[str, Any]] = []
    for index in sorted(bucket):
        item = bucket[index]
        msg: Dict[str, Any] = {}

        if "role" in item:
            msg["role"] = str(item["role"])
        if "content" in item:
            msg["content"] = str(item["content"])

        raw_tool_calls = item.get("tool_calls") or item.get("toolCalls")
        if raw_tool_calls is not None:
            parsed_tool_calls = maybe_parse_json(raw_tool_calls)
            if parsed_tool_calls:
                msg["tool_calls"] = parsed_tool_calls

        if msg:
            messages.append(msg)

    return messages


def normalize_llm_output(raw_output: Any) -> Any:
    parsed = maybe_parse_json(raw_output)
    if isinstance(parsed, (dict, list, str)):
        return parsed
    return raw_output


def llm_amp(attrs: Dict[str, Any]) -> Tuple[Any, Any, Dict[str, Any]]:
    input_messages = extract_message_blocks(attrs, "llm.input_messages")
    output_messages = extract_message_blocks(attrs, "llm.output_messages")

    raw_input_value = maybe_parse_json(attrs.get("input.value"))
    raw_output_value = normalize_llm_output(attrs.get("output.value"))

    amp_input = input_messages if input_messages else raw_input_value
    amp_output = output_messages if output_messages else raw_output_value

    data: Dict[str, Any] = {}

    model = attrs.get("llm.model_name")
    if isinstance(model, str) and model.strip():
        data["model"] = model.strip()

    vendor = attrs.get("llm.vendor")
    if isinstance(vendor, str) and vendor.strip():
        data["vendor"] = vendor.strip()

    temperature = parse_temperature(attrs)
    if temperature is not None:
        data["temperature"] = temperature

    tools = extract_llm_tools(attrs)
    if tools:
        data["tools"] = tools

    token_usage = extract_token_usage(attrs)
    if token_usage:
        data["tokenUsage"] = token_usage

    return amp_input, amp_output, data


def tool_amp(attrs: Dict[str, Any]) -> Tuple[Any, Any, Dict[str, Any]]:
    amp_input = maybe_parse_json(attrs.get("input.value"))
    amp_output = maybe_parse_json(attrs.get("output.value"))

    data: Dict[str, Any] = {}
    name = attrs.get("tool.name")
    if isinstance(name, str) and name.strip():
        data["name"] = name.strip()

    description = attrs.get("tool.description")
    if isinstance(description, str) and description.strip():
        data["description"] = description.strip()

    return amp_input, amp_output, data


def agent_amp(span: Dict[str, Any], attrs: Dict[str, Any]) -> Tuple[Any, Any, Dict[str, Any]]:
    amp_input = maybe_parse_json(attrs.get("input.value"))
    amp_output = maybe_parse_json(attrs.get("output.value"))

    data: Dict[str, Any] = {
        "name": span.get("span_name", "") or ""
    }

    model = attrs.get("llm.model_name")
    if isinstance(model, str) and model.strip():
        data["model"] = model.strip()

    token_usage = extract_token_usage(attrs)
    if token_usage:
        data["tokenUsage"] = token_usage

    data["framework"] = "openinference"

    tools_names = maybe_parse_json(attrs.get("smolagents.tools_names"))
    if isinstance(tools_names, list):
        tools = []
        for t in tools_names:
            name = str(t).strip()
            if name:
                tools.append({"name": name})
        if tools:
            data["tools"] = tools

    max_steps = to_int(attrs.get("smolagents.max_steps"))
    if max_steps is not None:
        data["maxIter"] = max_steps

    return amp_input, amp_output, data


def retriever_amp(attrs: Dict[str, Any]) -> Tuple[Any, Any, Dict[str, Any]]:
    amp_input = maybe_parse_json(attrs.get("input.value"))
    amp_output = maybe_parse_json(attrs.get("output.value"))

    data: Dict[str, Any] = {}

    vector_db = attrs.get("retrieval.vector_db") or attrs.get("vector_db")
    if isinstance(vector_db, str) and vector_db.strip():
        data["vectorDB"] = vector_db.strip()

    top_k = to_int(attrs.get("retrieval.top_k") or attrs.get("top_k"))
    if top_k is not None:
        data["topK"] = top_k

    return amp_input, amp_output, data


def chain_amp(attrs: Dict[str, Any]) -> Tuple[Any, Any, Dict[str, Any]]:
    return maybe_parse_json(attrs.get("input.value")), maybe_parse_json(attrs.get("output.value")), {}


def compute_amp_attributes(span: Dict[str, Any]) -> Dict[str, Any]:
    attrs = span.get("span_attributes")
    if not isinstance(attrs, dict):
        attrs = {}

    kind = infer_kind(str(span.get("span_name", "")), attrs)
    status, error_obj = extract_status_and_error(span, attrs)

    amp_input = None
    amp_output = None
    data: Dict[str, Any] = {}

    if kind == "llm":
        amp_input, amp_output, data = llm_amp(attrs)
    elif kind == "tool":
        amp_input, amp_output, data = tool_amp(attrs)
    elif kind == "agent":
        amp_input, amp_output, data = agent_amp(span, attrs)
    elif kind == "retriever":
        amp_input, amp_output, data = retriever_amp(attrs)
    elif kind == "chain":
        amp_input, amp_output, data = chain_amp(attrs)
    else:
        amp_input = maybe_parse_json(attrs.get("input.value"))
        amp_output = maybe_parse_json(attrs.get("output.value"))

    result: Dict[str, Any] = {
        "kind": kind,
        "status": status,
    }

    if error_obj:
        result["error"] = error_obj
    if amp_input not in (None, "", [], {}):
        result["input"] = amp_input
    if amp_output not in (None, "", [], {}):
        result["output"] = amp_output
    if data:
        result["data"] = data

    return result


def process_span_recursive(span: Dict[str, Any]) -> int:
    span["ampAttributes"] = compute_amp_attributes(span)
    count = 1

    children = span.get("child_spans")
    if isinstance(children, list):
        for child in children:
            if isinstance(child, dict):
                count += process_span_recursive(child)

    return count


def process_trace_record(row) -> Tuple[str, Dict[str, Any], int]:
    trace_obj = maybe_parse_json(row["trace"])
    if not isinstance(trace_obj, dict):
        raise ValueError("row['trace'] is not a valid JSON object")

    total_spans = 0
    spans = trace_obj.get("spans")
    if isinstance(spans, list):
        for span in spans:
            if isinstance(span, dict):
                total_spans += process_span_recursive(span)

    trace_id = str(trace_obj.get("trace_id") or row.name)
    record = {
        "trace_id": trace_id,
        "labels": maybe_parse_json(row["labels"]) if "labels" in row and row["labels"] is not None else None,
        "trace": trace_obj,
    }
    return trace_id, record, total_spans


file_count = 0
span_count = 0

for _, row in df_gaia.iterrows():
    trace_id, record, n_spans = process_trace_record(row)
    out_path = PREPROCESSED_DIR / f"{trace_id}.json"
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)
        f.write("\n")

    file_count += 1
    span_count += n_spans

print(f"Processed {file_count} traces")
print(f"Processed {span_count} spans")
print(f"Saved preprocessed traces to: {PREPROCESSED_DIR}")


Processed 117 traces
Processed 3579 spans
Saved preprocessed traces to: /kaggle/working/preprocessed_traces


In [7]:
import json
from pathlib import Path

sample_file = sorted(Path("/kaggle/working/preprocessed_traces").glob("*.json"))[0]
obj = json.load(open(sample_file))
print(obj.keys())
print(obj["trace"].keys())
print(obj["trace"]["spans"][0].keys())
print(obj["trace"]["spans"][0].get("ampAttributes"))


dict_keys(['trace_id', 'labels', 'trace'])
dict_keys(['trace_id', 'spans'])
dict_keys(['timestamp', 'trace_id', 'span_id', 'parent_span_id', 'trace_state', 'span_name', 'span_kind', 'service_name', 'resource_attributes', 'scope_name', 'scope_version', 'span_attributes', 'duration', 'status_code', 'status_message', 'events', 'links', 'logs', 'child_spans', 'ampAttributes'])
{'kind': 'unknown', 'status': {'error': False}}


## Load preprocessed traces, flatten spans, and build AMP typed Trace objects in memory

This does not write typed spans to disk. It creates a Python list called traces.



In [8]:
import json
import pandas as pd
from pathlib import Path

from amp_evaluation.trace.fetcher import _parse_trace
from amp_evaluation.trace.parser import parse_trace_for_evaluation

PREPROCESSED_DIR = Path("/kaggle/working/preprocessed_traces")


def _to_iso_z(ts) -> str:
    if ts is None:
        return pd.Timestamp.utcnow().isoformat().replace("+00:00", "Z")
    return pd.Timestamp(ts).isoformat().replace("+00:00", "Z")


def _duration_to_nanos(raw_duration) -> int:
    if not raw_duration:
        return 0
    try:
        return int(pd.Timedelta(raw_duration).value)
    except Exception:
        return 0


def flatten_trace_spans(root_spans):
    flat = []

    def walk(span):
        if not isinstance(span, dict):
            return

        start_time = _to_iso_z(span.get("timestamp"))
        duration_ns = _duration_to_nanos(span.get("duration"))
        end_time = (
            pd.Timestamp(start_time) + pd.to_timedelta(duration_ns, unit="ns")
        ).isoformat().replace("+00:00", "Z")

        flat.append({
            "traceId": str(span.get("trace_id", "")),
            "spanId": str(span.get("span_id", "")),
            "parentSpanId": span.get("parent_span_id"),
            "name": span.get("span_name", "") or "",
            "service": span.get("service_name", "") or "",
            "startTime": start_time,
            "endTime": end_time,
            "durationInNanos": duration_ns,
            "kind": str(span.get("span_kind", "INTERNAL")).upper(),
            "status": str(span.get("status_code", "UNSET")).upper(),
            "attributes": span.get("span_attributes", {}) or {},
            "ampAttributes": span.get("ampAttributes", {}) or {},
        })

        for child in span.get("child_spans") or []:
            walk(child)

    for span in root_spans or []:
        walk(span)

    flat.sort(key=lambda s: (s.get("startTime") or "", s.get("spanId") or ""))
    return flat


def infer_trace_io(trace_obj: dict, flat_spans: list[dict]) -> tuple[str, str, str]:
    trace_id = str(trace_obj.get("trace_id") or (flat_spans[0]["traceId"] if flat_spans else ""))

    trace_input = trace_obj.get("input", trace_obj.get("question", ""))
    trace_output = trace_obj.get("output", trace_obj.get("final_answer", ""))

    if not trace_input:
        for span in flat_spans:
            amp = span.get("ampAttributes") or {}
            raw_input = amp.get("input")
            if isinstance(raw_input, str) and raw_input.strip():
                trace_input = raw_input
                break
            if isinstance(raw_input, list) and raw_input:
                for msg in raw_input:
                    if isinstance(msg, dict) and msg.get("role") == "user" and msg.get("content"):
                        trace_input = msg["content"]
                        break
                if trace_input:
                    break

    if not trace_output:
        for span in reversed(flat_spans):
            amp = span.get("ampAttributes") or {}
            raw_output = amp.get("output")
            if isinstance(raw_output, str) and raw_output.strip():
                trace_output = raw_output
                break
            if isinstance(raw_output, dict) and raw_output.get("content"):
                trace_output = raw_output["content"]
                break
            if isinstance(raw_output, list) and raw_output:
                last = raw_output[-1]
                if isinstance(last, dict) and last.get("content"):
                    trace_output = last["content"]
                    break

    return trace_id, str(trace_input or ""), str(trace_output or "")


traces = []
files = sorted(PREPROCESSED_DIR.glob("*.json"))

for path in files:
    with path.open("r", encoding="utf-8") as f:
        record = json.load(f)

    trace_obj = record["trace"]
    flat_spans = flatten_trace_spans(trace_obj.get("spans", []))
    trace_id, trace_input, trace_output = infer_trace_io(trace_obj, flat_spans)

    if not flat_spans:
        continue

    error_count = sum(
        1
        for span in flat_spans
        if ((span.get("ampAttributes") or {}).get("status") or {}).get("error")
        or span.get("status") == "ERROR"
    )

    api_trace = {
        "traceId": trace_id,
        "rootSpanId": flat_spans[0]["spanId"],
        "rootSpanName": flat_spans[0]["name"],
        "startTime": flat_spans[0]["startTime"],
        "endTime": max(span["endTime"] for span in flat_spans),
        "spans": flat_spans,
        "rootSpanKind": flat_spans[0]["kind"],
        "durationInNanos": sum(span["durationInNanos"] for span in flat_spans),
        "spanCount": len(flat_spans),
        "status": {"errorCount": error_count},
        "input": trace_input,
        "output": trace_output,
    }

    otel_trace = _parse_trace(api_trace)
    parsed_trace = parse_trace_for_evaluation(otel_trace)
    parsed_trace._labels = record.get("labels")
    traces.append(parsed_trace)

print(f"Built {len(traces)} AMP Trace objects in memory")

if traces:
    sample_trace = traces[0]
    print("Sample parsed trace:")
    print("  trace_id       :", sample_trace.trace_id)
    print("  total spans    :", len(sample_trace.spans))
    print("  agent spans    :", len(sample_trace.get_agents()))
    print("  tool spans     :", len(sample_trace.get_tool_calls()))
    print("  retrieval spans:", len(sample_trace.get_retrievals()))


Built 117 AMP Trace objects in memory
Sample parsed trace:
  trace_id       : 0035f455b3ff2295167a844f04d85d34
  total spans    : 10
  agent spans    : 2
  tool spans     : 1
  retrieval spans: 0


## Cell 6 — Define Evaluators Using AMP SDK


In [9]:
from amp_evaluation import builtin

EVALUATORS = [
    builtin("helpfulness"),
    builtin("accuracy"),
    builtin("groundedness"),
    builtin("instruction_following"),
    builtin("reasoning_quality"),
]

print("Configured evaluators:\n")

for i, e in enumerate(EVALUATORS, 1):
    info = e.info
    print(f"{i}. {info.name}")
    print(f"   level       : {info.level}")
    print(f"   description : {info.description}")
    print()


Configured evaluators:

1. helpfulness
   level       : trace
   description : Scores whether the response actually helps the user with what they asked for. Checks for actionable, useful content vs empty acknowledgments.

2. accuracy
   level       : trace
   description : Scores factual correctness of information in the response using the LLM's own knowledge. Does not use tool or retrieval evidence.

3. groundedness
   level       : trace
   description : Verifies that factual claims in the response are grounded in tool results or retrieved documents. Skips when no evidence is available (configurable via on_missing_context).

4. instruction_following
   level       : agent
   description : Checks whether the agent follows system prompt constraints and user instructions. Runs per agent. Always evaluates since user input is always available.

5. reasoning_quality
   level       : agent
   description : Scores whether the agent's execution steps are logical, purposeful, and well-reasoned

# LlaMa

## cell 1: configs

In [10]:
from pathlib import Path

LOCAL_JUDGE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
LOAD_IN_4BIT = True
MAX_SEQ_LENGTH = 50000
MAX_NEW_TOKENS = 160
MAX_RETRIES = 2
TRIAL_TIME_LIMIT_SECONDS = None
TRACE_LIMIT = None
RESET_OUTPUTS = True


TRIAL_RESULTS_PATH = Path("/kaggle/working/llama_3_1_8b_instruct_trial_5_results.csv")
TRIAL_DEBUG_LOG_PATH = Path("/kaggle/working/llama_3_1_8b_instruct_trial_5_failures.jsonl")

CSV_COLUMNS = [
    "model",
    "trace_id",
    "evaluator",
    "level",
    "score",
    "explanation",
    "latency_ms",
    "is_skipped",
    "skip_reason",
]

SHORT_OUTPUT_INSTRUCTIONS = """
Respond with ONLY a JSON object:
{
  "explanation": "<brief explanation in 1-3 short sentences>",
  "score": <float between 0.0 and 1.0>
}
Do not include markdown fences.
Do not include any extra text before or after the JSON.
Keep the explanation concise.
"""

print(f"Local judge model: {LOCAL_JUDGE_MODEL}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print(f"Max retries: {MAX_RETRIES}")
print(f"Trial time limit (s): {TRIAL_TIME_LIMIT_SECONDS}")
print(f"Trace limit: {TRACE_LIMIT}")
print(f"Results path: {TRIAL_RESULTS_PATH}")
print(f"Debug log path: {TRIAL_DEBUG_LOG_PATH}")
print("Using shortened local SLM output instructions.")


Local judge model: meta-llama/Meta-Llama-3.1-8B-Instruct
Max new tokens: 160
Max retries: 2
Trial time limit (s): None
Trace limit: None
Results path: /kaggle/working/llama_3_1_8b_instruct_trial_5_results.csv
Debug log path: /kaggle/working/llama_3_1_8b_instruct_trial_5_failures.jsonl
Using shortened local SLM output instructions.


## Cell 2: helper

In [11]:
from unsloth import FastLanguageModel
import csv
import gc
import json
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import torch
from tqdm.auto import tqdm

from amp_evaluation import builtin
from amp_evaluation.models import EvalResult
from amp_evaluation.trace import AgentTrace

EVALUATOR_NAMES = [
    "helpfulness",
    "accuracy",
    "groundedness",
    "instruction_following",
    "reasoning_quality",
]

@dataclass(frozen=True)
class EvalJob:
    trace_id: str
    evaluator_name: str
    level: str
    occurrence_index: int
    prompt: Optional[str]
    evaluator: object
    trace: object
    target: object
    target_label: str
    precomputed_result: Optional[EvalResult] = None

    @property
    def base_key(self) -> Tuple[str, str, str]:
        return (self.trace_id, self.evaluator_name, self.level)

def load_local_judge(model_name: str):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
    )
    FastLanguageModel.for_inference(model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    return model, tokenizer

def cleanup_model(model=None, tokenizer=None):
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def render_prompt_text(tokenizer, prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
    return prompt

def run_generation(model, tokenizer, prompt: str) -> str:
    rendered_prompt = render_prompt_text(tokenizer, prompt)
    tokenized = tokenizer(
        text=rendered_prompt,
        return_tensors="pt",
        truncation=True,
    )
    model_inputs = {
        key: value.to(model.device)
        for key, value in tokenized.items()
        if key in {"input_ids", "attention_mask"}
    }
    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prompt_length = model_inputs["input_ids"].shape[1]
    generated = outputs[0][prompt_length:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def build_evaluators():
    evaluators = []
    for name in EVALUATOR_NAMES:
        ev = builtin(name)
        ev.model = LOCAL_JUDGE_MODEL
        ev.max_retries = MAX_RETRIES
        ev._OUTPUT_INSTRUCTIONS = SHORT_OUTPUT_INSTRUCTIONS
        evaluators.append(ev)
    return evaluators

def build_full_prompt(evaluator, target, task=None) -> str:
    prompt = evaluator._dispatch_build_prompt(target, task)
    return prompt + evaluator._OUTPUT_INSTRUCTIONS

def precompute_result_if_needed(evaluator, trace) -> Optional[EvalResult]:
    if evaluator.name != "groundedness":
        return None
    if trace.get_tool_calls() or trace.get_retrievals():
        return None
    if getattr(evaluator, "on_missing_context", "skip") == "zero":
        return EvalResult(
            score=0.0,
            passed=False,
            explanation="No tool or retrieval spans found; cannot assess groundedness",
        )
    return EvalResult.skip("No tool or retrieval spans found in this trace")

def build_jobs(traces, evaluators) -> List[EvalJob]:
    jobs = []
    occurrences: Dict[Tuple[str, str, str], int] = defaultdict(int)

    for trace in traces:
        for ev in evaluators:
            level = ev.level.value

            if level == "trace":
                precomputed = precompute_result_if_needed(ev, trace)
                prompt = None if precomputed is not None else build_full_prompt(ev, trace)
                base_key = (trace.trace_id, ev.name, level)
                occurrence_index = occurrences[base_key]
                occurrences[base_key] += 1
                jobs.append(
                    EvalJob(
                        trace_id=trace.trace_id,
                        evaluator_name=ev.name,
                        level=level,
                        occurrence_index=occurrence_index,
                        prompt=prompt,
                        evaluator=ev,
                        trace=trace,
                        target=trace,
                        target_label="trace",
                        precomputed_result=precomputed,
                    )
                )
                continue

            if level != "agent":
                raise ValueError(f"Unsupported evaluator level: {level}")

            agent_spans = trace.get_agents()
            targets = []

            if not agent_spans:
                root_span = trace._get_root_span()
                fallback_agent_id = root_span.span_id if root_span else trace.trace_id
                fallback = AgentTrace(
                    agent_id=fallback_agent_id,
                    input=trace.input,
                    output=trace.output,
                    steps=trace._get_agent_steps(deduplicate_messages=True),
                    metrics=trace.metrics,
                )
                targets.append((f"fallback-agent:{fallback_agent_id}", fallback))
            else:
                for agent_span in agent_spans:
                    agent_trace = trace._create_agent_trace(agent_span.span_id)
                    targets.append((f"agent:{agent_trace.agent_id}", agent_trace))

            for target_label, agent_trace in targets:
                base_key = (trace.trace_id, ev.name, level)
                occurrence_index = occurrences[base_key]
                occurrences[base_key] += 1
                jobs.append(
                    EvalJob(
                        trace_id=trace.trace_id,
                        evaluator_name=ev.name,
                        level=level,
                        occurrence_index=occurrence_index,
                        prompt=build_full_prompt(ev, agent_trace),
                        evaluator=ev,
                        trace=trace,
                        target=agent_trace,
                        target_label=target_label,
                    )
                )

    return jobs

def build_retry_prompt(base_prompt: str, attempt: int, last_error: Optional[str]) -> str:
    if attempt <= 0 or not last_error:
        return base_prompt

    retry_ctx = (
        f"\n\n[IMPORTANT: Your previous response was invalid: {last_error}. "
        "You MUST respond with ONLY a JSON object containing exactly two fields:\n"
        '{"explanation": "<your analysis>", "score": <float between 0.0 and 1.0>}\n'
        "The 'score' MUST be a top-level numeric field in the JSON, NOT embedded in the explanation text.]"
    )
    return base_prompt + retry_ctx

def init_output_artifacts(reset_outputs: bool = True):
    if reset_outputs:
        for path in (TRIAL_RESULTS_PATH, TRIAL_DEBUG_LOG_PATH):
            if path.exists():
                path.unlink()

    TRIAL_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

    with TRIAL_RESULTS_PATH.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS)
        writer.writeheader()

    TRIAL_DEBUG_LOG_PATH.touch()

def append_result_row(row: Dict[str, object]):
    with TRIAL_RESULTS_PATH.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS)
        writer.writerow(row)

def append_failure_log(payload: Dict[str, object]):
    with TRIAL_DEBUG_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")

def result_to_row(job: EvalJob, result: EvalResult, latency_ms: float) -> Dict[str, object]:
    return {
        "model": LOCAL_JUDGE_MODEL,
        "trace_id": job.trace_id,
        "evaluator": job.evaluator_name,
        "level": job.level,
        "score": "" if result.is_skipped else result.score,
        "explanation": "" if result.is_skipped else (result.explanation or ""),
        "latency_ms": round(latency_ms, 2),
        "is_skipped": result.is_skipped,
        "skip_reason": result.skip_reason or "",
    }

def finalize_retry_failure(evaluator, last_error: Optional[str]) -> EvalResult:
    max_retries = int(getattr(evaluator, "max_retries", 2))
    return EvalResult.skip(
        f"LLM judge failed after {max_retries + 1} attempts: {last_error or 'Unknown error'} [model={LOCAL_JUDGE_MODEL}]"
    )


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [12]:
model, tokenizer = load_local_judge(LOCAL_JUDGE_MODEL)
print(next(model.parameters()).dtype)


==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


torch.float16


In [13]:
cleanup_model(model, tokenizer)


## Cell 3: Trial runner





In [14]:
def run_llama_trial(traces, model, tokenizer):
    traces_to_run = traces[:TRACE_LIMIT] if TRACE_LIMIT is not None else traces
    evaluators = build_evaluators()
    jobs = build_jobs(traces_to_run, evaluators)
    init_output_artifacts(reset_outputs=RESET_OUTPUTS)

    print(f"Model: {LOCAL_JUDGE_MODEL}")
    print(f"Trace limit: {TRACE_LIMIT}")
    print(f"Traces available: {len(traces_to_run)}")
    print(f"Jobs queued: {len(jobs)}")
    print(f"Trial time limit: {TRIAL_TIME_LIMIT_SECONDS}")
    print(f"Results path: {TRIAL_RESULTS_PATH}")
    print(f"Debug log path: {TRIAL_DEBUG_LOG_PATH}")

    start_time = time.time()
    attempted_jobs = 0
    success_count = 0
    skip_count = 0
    per_evaluator = Counter()
    per_evaluator_skips = Counter()

    progress = tqdm(total=len(jobs), desc="Llama trial run (5 traces)", unit="job")

    try:
        for job in jobs:
            elapsed = time.time() - start_time
            if TRIAL_TIME_LIMIT_SECONDS is not None and elapsed >= TRIAL_TIME_LIMIT_SECONDS:
                print(f"Reached trial time limit after {elapsed:.1f}s; stopping.")
                break

            attempted_jobs += 1
            per_evaluator[job.evaluator_name] += 1

            if job.precomputed_result is not None:
                result = job.precomputed_result
                append_result_row(result_to_row(job, result, 0.0))
                if result.is_skipped:
                    skip_count += 1
                    per_evaluator_skips[job.evaluator_name] += 1
                    progress.set_postfix(status="pre-skip", evaluator=job.evaluator_name)
                else:
                    success_count += 1
                    progress.set_postfix(status="pre-ok", evaluator=job.evaluator_name)
                progress.update(1)

                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue

            parsed_result = None
            last_error = None
            total_latency_ms = 0.0
            last_raw_output = ""
            max_retries = int(getattr(job.evaluator, "max_retries", 2))

            for attempt in range(max_retries + 1):
                prompt = build_retry_prompt(job.prompt or "", attempt, last_error)
                t0 = time.perf_counter()
                try:
                    raw_output = run_generation(model, tokenizer, prompt)
                    last_raw_output = raw_output
                    elapsed_ms = (time.perf_counter() - t0) * 1000.0
                    total_latency_ms += elapsed_ms
                    parsed_result, last_error = job.evaluator._parse_and_validate(raw_output)
                except Exception as exc:
                    raw_output = ""
                    last_raw_output = raw_output
                    elapsed_ms = (time.perf_counter() - t0) * 1000.0
                    total_latency_ms += elapsed_ms
                    parsed_result = None
                    last_error = str(exc)

                if parsed_result is not None:
                    break

                append_failure_log({
                    "trace_id": job.trace_id,
                    "evaluator": job.evaluator_name,
                    "level": job.level,
                    "target_label": job.target_label,
                    "attempt": attempt + 1,
                    "error": last_error,
                    "raw_output": last_raw_output,
                })

            if parsed_result is None:
                parsed_result = finalize_retry_failure(job.evaluator, last_error)

            append_result_row(result_to_row(job, parsed_result, total_latency_ms))

            if parsed_result.is_skipped:
                skip_count += 1
                per_evaluator_skips[job.evaluator_name] += 1
                progress.set_postfix(status="skip", evaluator=job.evaluator_name)
            else:
                success_count += 1
                progress.set_postfix(status="ok", evaluator=job.evaluator_name)

            progress.update(1)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    finally:
        progress.close()

    summary = {
        "model": LOCAL_JUDGE_MODEL,
        "trace_limit": TRACE_LIMIT,
        "attempted_jobs": attempted_jobs,
        "success_count": success_count,
        "skip_count": skip_count,
        "elapsed_seconds": round(time.time() - start_time, 2),
        "per_evaluator": dict(per_evaluator),
        "per_evaluator_skips": dict(per_evaluator_skips),
        "results_path": str(TRIAL_RESULTS_PATH),
        "debug_log_path": str(TRIAL_DEBUG_LOG_PATH),
    }
    return summary


In [15]:
import warnings
from transformers.utils import logging

# Hide HF generation warnings/info messages
logging.set_verbosity_error()

# Hide the max_new_tokens vs max_length warning
warnings.filterwarnings(
    "ignore",
    message=r".*max_new_tokens.*max_length.*",
)

# Hide the deprecated attention mask warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r".*AttentionMaskConverter.*deprecated.*",
)


In [16]:
#model, tokenizer = load_local_judge(LOCAL_JUDGE_MODEL)
trial_summary = run_llama_trial(traces, model, tokenizer)
print(json.dumps(trial_summary, indent=2))
cleanup_model(model, tokenizer)


Model: meta-llama/Meta-Llama-3.1-8B-Instruct
Trace limit: None
Traces available: 117
Jobs queued: 939
Trial time limit: None
Results path: /kaggle/working/llama_3_1_8b_instruct_trial_5_results.csv
Debug log path: /kaggle/working/llama_3_1_8b_instruct_trial_5_failures.jsonl


Llama trial run (5 traces):   0%|          | 0/939 [00:00<?, ?job/s]

{
  "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
  "trace_limit": null,
  "attempted_jobs": 939,
  "success_count": 865,
  "skip_count": 74,
  "elapsed_seconds": 27387.47,
  "per_evaluator": {
    "helpfulness": 117,
    "accuracy": 117,
    "groundedness": 117,
    "instruction_following": 294,
    "reasoning_quality": 294
  },
  "per_evaluator_skips": {
    "instruction_following": 41,
    "reasoning_quality": 32,
    "groundedness": 1
  },
  "results_path": "/kaggle/working/llama_3_1_8b_instruct_trial_5_results.csv",
  "debug_log_path": "/kaggle/working/llama_3_1_8b_instruct_trial_5_failures.jsonl"
}


## Cell 4: Trial inspect





In [17]:
import json
import pandas as pd
from collections import Counter

df_trial = pd.read_csv(TRIAL_RESULTS_PATH)
print(f"Trial rows: {len(df_trial)}")
print(f"Non-skipped rows: {(df_trial['is_skipped'] != True).sum()}")
print(f"Skipped rows: {(df_trial['is_skipped'] == True).sum()}")
print()
print("Per-evaluator rows:")
print(df_trial.groupby(["evaluator", "is_skipped"]).size())
print()

skip_reasons = Counter(df_trial.loc[df_trial["is_skipped"] == True, "skip_reason"].fillna(""))
print("Top skip reasons:")
for reason, count in skip_reasons.most_common(10):
    print(f"- {count}x {reason[:220]}")
print()

failure_records = []
with TRIAL_DEBUG_LOG_PATH.open() as handle:
    for line in handle:
        line = line.strip()
        if line:
            failure_records.append(json.loads(line))

print(f"Logged raw failure records: {len(failure_records)}")
if failure_records:
    print("Sample failure summary:")
    sample = failure_records[0]
    print({k: sample[k] for k in ["trace_id", "evaluator", "level", "target_label", "attempt"]})
    print("Error:", str(sample.get("error", ""))[:500])
    print("Raw output preview:", str(sample.get("raw_output", ""))[:1000])

display(df_trial.head(10))


Trial rows: 939
Non-skipped rows: 865
Skipped rows: 74

Per-evaluator rows:
evaluator              is_skipped
accuracy               False         117
groundedness           False         116
                       True            1
helpfulness            False         117
instruction_following  False         253
                       True           41
reasoning_quality      False         262
                       True           32
dtype: int64

Top skip reasons:
- 3x LLM judge failed after 3 attempts: 1 validation error for JudgeOutput
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value="The agent's reasoning is questionable choices.", input_type=str]
  
- 2x LLM judge failed after 3 attempts: 1 validation error for JudgeOutput
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='The final answer is: {"e...swer is:", "score": 0.0', input_type=str
- 1x LLM judge failed after 3 attempts: 1 validation error for JudgeOutput
  I

,model,trace_id,evaluator,level,score,explanation,latency_ms,is_skipped,skip_reason
0,meta-llama/Meta-Llama-3.1-8B-Instruct,0035f455b3ff2295167a844f04d85d34,helpfulness,trace,0.00,"The response does not address the question, wh...",9384.82,False,NaN
1,meta-llama/Meta-Llama-3.1-8B-Instruct,0035f455b3ff2295167a844f04d85d34,accuracy,trace,0.25,"The response is a single zip code, which is no...",4030.43,False,NaN
2,meta-llama/Meta-Llama-3.1-8B-Instruct,0035f455b3ff2295167a844f04d85d34,groundedness,trace,0.00,The agent's response is unsupported by the pro...,4511.52,False,NaN
3,meta-llama/Meta-Llama-3.1-8B-Instruct,0035f455b3ff2295167a844f04d85d34,instruction_following,agent,0.00,"No instructions were provided, so the agent ha...",2130.18,False,NaN
4,meta-llama/Meta-Llama-3.1-8B-Instruct,0035f455b3ff2295167a844f04d85d34,instruction_following,agent,0.00,The agent failed to provide a correct answer t...,11273.45,False,NaN
5,meta-llama/Meta-Llama-3.1-8B-Instruct,0035f455b3ff2295167a844f04d85d34,reasoning_quality,agent,0.00,"No execution steps were provided, making it im...",2211.91,False,NaN
6,meta-llama/Meta-Llama-3.1-8B-Instruct,0035f455b3ff2295167a844f04d85d34,reasoning_quality,agent,0.50,"The agent's reasoning is mostly logical, but i...",9924.97,False,NaN
7,meta-llama/Meta-Llama-3.1-8B-Instruct,0140b3f657eddf76ca82f72c49ac8e58,helpfulness,trace,0.25,The response provides the surname of the equin...,4417.00,False,NaN
8,meta-llama/Meta-Llama-3.1-8B-Instruct,0140b3f657eddf76ca82f72c49ac8e58,accuracy,trace,0.00,The response claims that the surname of the eq...,5966.36,False,NaN
9,meta-llama/Meta-Llama-3.1-8B-Instruct,0140b3f657eddf76ca82f72c49ac8e58,groundedness,trace,0.00,The response claims that the surname of the eq...,21988.92,False,NaN


In [18]:
import json
import re
import pandas as pd
from collections import Counter

failure_records = []
with TRIAL_DEBUG_LOG_PATH.open() as handle:
    for line in handle:
        line = line.strip()
        if line:
            failure_records.append(json.loads(line))

def classify_failure(raw):
    raw = (raw or "").strip()

    if not raw:
        return "empty"

    if raw.startswith("{") and '"score"' in raw and '"explanation"' in raw:
        if raw.count("{") >= 1 and raw.count("}") == 0:
            return "truncated_json"
        return "near_json_malformed"

    if raw.startswith("###") or "Facts given in the task" in raw:
        return "role_confusion_task_answer"

    if raw.startswith("```"):
        return "markdown_wrapped_json_or_text"

    if re.fullmatch(r"(possible[\s\n]*){5,}", raw.lower()):
        return "degenerate_repetition"

    if len(set(raw.split())) < max(3, len(raw.split()) * 0.2):
        return "degenerate_low_diversity"

    if '"score"' in raw or '"explanation"' in raw:
        return "near_json_malformed"

    return "plain_prose_or_other"

df_fail = pd.DataFrame(failure_records)
df_fail["failure_type"] = df_fail["raw_output"].apply(classify_failure)

print(df_fail["failure_type"].value_counts())
display(df_fail[["trace_id", "evaluator", "level", "attempt", "failure_type", "raw_output"]].head(20))


failure_type
empty                         85
plain_prose_or_other          68
role_confusion_task_answer    67
near_json_malformed           23
degenerate_low_diversity      19
Name: count, dtype: int64


,trace_id,evaluator,level,attempt,failure_type,raw_output
0,0140b3f657eddf76ca82f72c49ac8e58,groundedness,trace,1,role_confusion_task_answer,### 1. Facts given in the task\n- The task is ...
1,0140b3f657eddf76ca82f72c49ac8e58,instruction_following,agent,1,plain_prose_or_other,The agent received the following instructions:...
2,0140b3f657eddf76ca82f72c49ac8e58,instruction_following,agent,2,plain_prose_or_other,The agent received the following instructions:...
3,0140b3f657eddf76ca82f72c49ac8e58,instruction_following,agent,3,plain_prose_or_other,The agent received the following instructions:...
4,0140b3f657eddf76ca82f72c49ac8e58,instruction_following,agent,1,plain_prose_or_other,I will follow the instructions to the letter.
5,0140b3f657eddf76ca82f72c49ac8e58,instruction_following,agent,2,plain_prose_or_other,I will follow the instructions to the letter.
6,0140b3f657eddf76ca82f72c49ac8e58,instruction_following,agent,3,plain_prose_or_other,I will follow the instructions to the letter.
7,0140b3f657eddf76ca82f72c49ac8e58,reasoning_quality,agent,1,plain_prose_or_other,The agent's reasoning is incoherent; steps are...
8,0140b3f657eddf76ca82f72c49ac8e58,reasoning_quality,agent,2,plain_prose_or_other,The agent's reasoning is unclear and does not ...
9,0140b3f657eddf76ca82f72c49ac8e58,reasoning_quality,agent,3,plain_prose_or_other,The agent's reasoning is mostly logical and pu...


In [19]:
evaluators = build_evaluators()
jobs = build_jobs(traces[:5], evaluators)

job_map = {}
for job in jobs:
    key = (job.trace_id, job.evaluator_name, job.level, job.target_label)
    job_map[key] = job

for _, row in df_fail.head(5).iterrows():
    key = (row["trace_id"], row["evaluator"], row["level"], row["target_label"])
    job = job_map.get(key)
    if job and job.prompt:
        print("=" * 120)
        print("trace_id:", row["trace_id"])
        print("evaluator:", row["evaluator"])
        print("level:", row["level"])
        print("target_label:", row["target_label"])
        print("PROMPT PREVIEW:")
        print(job.prompt[:4000])
        print("\nRAW OUTPUT PREVIEW:")
        print((row.get("raw_output") or "")[:2000])
        break


trace_id: 0140b3f657eddf76ca82f72c49ac8e58
evaluator: groundedness
level: trace
target_label: trace
PROMPT PREVIEW:
You are an expert evaluator. Your sole criterion is GROUNDEDNESS: are the factual claims in the response grounded in the evidence that was available to the agent?

User Query: Below I will present you a task.

You will now build a comprehensive preparatory survey of which facts we have at our disposal and which ones we still need.
To do so, you will have to read the task and identify things that must be discovered in order to successfully complete it.
Don't make any assumptions. For each item, provide a thorough reasoning. Here is how you will structure this survey:

---
### 1. Facts given in the task
List here the specific facts given in the task that could help you (there might be nothing here).

### 2. Facts to look up
List here any facts that we may need to look up.
Also list where to find each of these, for instance a website, a file... - maybe the task contains some